In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("ollama:gemma4")
standard_model = init_chat_model("ollama:gemma4:e2b")


# see docs: 
# request: https://reference.langchain.com/python/langchain/agents/middleware/types/ModelRequest
# response: https://reference.langchain.com/python/langchain/agents/middleware/types/ModelResponse
@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    # see docs: https://reference.langchain.com/python/langchain/agents/middleware/types/ModelRequest/override
    request = request.override(model=model)  

    return handler(request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:gemma4:e2b",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Oh, shoot, let me check... I think I did it right after lunch. Are you noticing any drooping, or is it looking okay?

If it needs more water, I can definitely make a note to do it first thing tomorrow! 😊


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

gemma4:e2b


In [7]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Oh, I'm not 100% sure about a date, as it depends on how fast it grows, but generally, we should look for a few signs.

Usually, if the roots start coming out of the drainage holes, or if the plant seems to be struggling because it's completely packed in, it's time for a slightly bigger pot. I can keep an eye out for those signs and let you know when it looks like we need to re-pot it! 😊


In [8]:
print(response["messages"][-1].response_metadata["model_name"])

gemma4
